# 초기설정

In [132]:
# 쌩 출력
import pandas as pd
PATH_TRAIN = "/content/drive/MyDrive/데이콘/open/train.csv"
train = train = pd.read_csv(PATH_TRAIN)
train

,ID,ip_src,port_src,ip_dst,port_dst,protocol,duration,pkt_count_fwd,pkt_count_bwd,rate_fwd_pkts,...,rate_bwd_bytes,payload_fwd_mean,payload_bwd_mean,tcp_win_fwd_init,tcp_win_bwd_init,tcp_syn_count,tcp_psh_count,tcp_rst_count,iat_avg_packets,attack_type
0,TRAIN_00000,192.168.10.18,3721.0,192.168.10.243,55.0,UDP,0.000231,2,2,8656.974200,...,1.142721e+06,81.000000,81.000000,0,0,0,0,0,NaN,Benign
1,TRAIN_00001,192.168.10.5,NaN,NaN,91.0,TCP,0.000000,0,1,0.000000,...,0.000000e+00,0.000000,0.000000,0,16392,0,0,0,1.499097e+09,Benign
2,TRAIN_00002,172.16.0.182,NaN,192.168.10.18,83.0,TCP,0.606002,11,5,18.151760,...,1.913360e+04,790.125000,790.125000,29200,28960,2,4,1,4.040012e-02,Hulk
3,TRAIN_00003,NaN,47668.0,192.168.10.18,NaN,TCP,1.003829,6,6,5.977114,...,1.155077e+04,993.416667,993.416667,29200,28960,3,2,0,9.125718e-02,Hulk
4,TRAIN_00004,192.168.10.5,51753.0,151.101.2.116,451.0,TCP,181.195271,62,89,NaN,...,6.896758e+02,NaN,NaN,8192,29200,2,22,0,NaN,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11994,TRAIN_11994,192.168.10.243,62329.0,192.168.10.249,49.0,UDP,0.023636,1,1,42.308586,...,5.542425e+03,91.000000,91.000000,0,0,0,0,0,2.363586e-02,Benign
11995,TRAIN_11995,192.168.10.243,60190.0,192.168.10.249,54.0,UDP,0.061452,1,1,16.272824,...,1.253007e+03,61.000000,61.000000,0,0,0,0,0,6.145215e-02,Benign
11996,TRAIN_11996,NaN,5741.0,NaN,NaN,UDP,0.049973,2,2,40.021603,...,3.641966e+03,66.000000,66.000000,0,0,0,0,0,1.665767e-02,Benign
11997,TRAIN_11997,NaN,56610.0,192.168.10.243,NaN,UDP,0.047832,2,2,41.813002,...,5.937446e+03,103.500000,103.500000,0,0,0,0,0,1.594400e-02,Benign


In [133]:
import pandas as pd
import matplotlib.pyplot as plt

# 경로 train,test
PATH_TRAIN = "/content/drive/MyDrive/데이콘/open/train.csv"
PATH_TEST = "/content/drive/MyDrive/데이콘/open/test.csv"

train = pd.read_csv(PATH_TRAIN)
test = pd.read_csv(PATH_TEST)

print("shape:",train.shape)
print("columns: ", train.columns)

shape: (11999, 22)
columns:  Index(['ID', 'ip_src', 'port_src', 'ip_dst', 'port_dst', 'protocol',
       'duration', 'pkt_count_fwd', 'pkt_count_bwd', 'rate_fwd_pkts',
       'rate_bwd_pkts', 'rate_fwd_bytes', 'rate_bwd_bytes', 'payload_fwd_mean',
       'payload_bwd_mean', 'tcp_win_fwd_init', 'tcp_win_bwd_init',
       'tcp_syn_count', 'tcp_psh_count', 'tcp_rst_count', 'iat_avg_packets',
       'attack_type'],
      dtype='object')


In [134]:
"""
train에 특정 class 증강 후 - 데이콘 1차 분석 참고
"""
print("Web_XSS 샘플 수 처음 :", (train['attack_type'] == 'Web_XSS').sum())

# 1. Web_XSS 샘플 필터링
web_xss_df = train[train['attack_type'] == 'Web_XSS']

# 2. Web_XSS 10배 복제
web_xss_augmented = pd.concat([web_xss_df] * 5, ignore_index=True)

# 3. 원본 train과 결합
train_augmented = pd.concat([train, web_xss_augmented], ignore_index=True)

# 4. 결과 확인
print(" 증강 완료")
print("Before:", train.shape)
print("After :", train_augmented.shape)
print("Web_XSS 샘플 수 증강 후:", (train_augmented['attack_type'] == 'Web_XSS').sum())

train = train_augmented.copy()

Web_XSS 샘플 수 처음 : 6
 증강 완료
Before: (11999, 22)
After : (12029, 22)
Web_XSS 샘플 수 증강 후: 36


In [135]:
train['attack_type'].value_counts()

,count
attack_type,
Benign,8791
Hulk,1719
Port_Scanning,793
DDoS,471
FTP_Brute_Force,47
GoldenEye,41
Web_XSS,36
Slow_HTTP,34
SSH_Brute_Force,30


# 전처리

참고
https://dacon.io/competitions/official/236502/codeshare/12514?page=1&dtype=recent

In [136]:
#Id 분리 (예측 대상 아님 + 나중에 써야함)
test_ids = test['ID']

In [137]:
# ==============================================================
# Cell 0 ▸ 공통 import & 설정
# ==============================================================
import pandas as pd, numpy as np, json, os
from sklearn.preprocessing import StandardScaler
pd.set_option("mode.chained_assignment", None)

EPS      = 1e-6             # 0 나눗셈 방지
PORT_BINS   = [-10, 0, 1024, 49152, 70000]
PORT_LABELS = ["missing", "well", "registered", "dynamic"]
MAIN_PORTS  = [21, 22, 53, 80, 443]
WINSOR_COLS = ["duration", "rate_fwd_pkts", "rate_bwd_pkts",
               "rate_fwd_bytes", "rate_bwd_bytes", "iat_avg_packets"]

def is_private(octet0):
    return octet0 in (10, 172, 192)


In [138]:
# ==============================================================
# Cell 2 ▸ 전처리 함수 (tree / nn 이원화)
# ==============================================================
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

def preprocess(
    df: pd.DataFrame,
    *,
    mode: str,                     # "tree" | "nn"
    clip_dict=None,
    scaler_params=None,
    is_train=True,
):
    """
    mode
      "tree" : NaN 유지, Winsor OFF (log1p만 적용), Scaling 없음
      "nn"   : NaN→mean Impute, Winsor 0.995, StandardScaler 적용
    """
    df = df.copy()

    # ── 1. 라벨 인코딩(Train 전용) ───────────────────────────────
    if is_train:
        lbl2id = {lbl: i for i, lbl in enumerate(df["attack_type"].unique())}
        df["target"] = df["attack_type"].map(lbl2id)
    else:
        lbl2id = None

    # ── 2. IP 옥텟 파생 ───────────────────────────────────────
    for side in ("src", "dst"):
        oct = df[f"ip_{side}"].str.split('.', expand=True)
        for i in range(4):
            df[f"ip_{side}_{i}"] = pd.to_numeric(oct[i], errors="coerce")
    df["same_subnet_24"] = (
        (df["ip_src_0"] == df["ip_dst_0"]) &
        (df["ip_src_1"] == df["ip_dst_1"]) &
        (df["ip_src_2"] == df["ip_dst_2"])
    )
    df["is_private_src"] = df["ip_src_0"].apply(is_private)
    df["is_private_dst"] = df["ip_dst_0"].apply(is_private)
    df["private_public"] = df["is_private_src"] & (~df["is_private_dst"])

    # ── 3. Port 처리 ───────────────────────────────────────────
    for col in ("port_src", "port_dst"):
        df[col] = df[col].fillna(-1)
        df[f"{col}_bucket"] = pd.cut(df[col], PORT_BINS, labels=PORT_LABELS)
    for p in MAIN_PORTS:
        df[f"is_port{p}"] = (df["port_src"] == p) | (df["port_dst"] == p)
    df["is_port_missing"] = df["port_dst"] == -1

    # ── 4. Payload 결측 구분 ──────────────────────────────────
    for col in ("payload_fwd_mean", "payload_bwd_mean"):
        df[f"is_{col}_missing"] = df[col].isna()
        df[f"is_{col}_zero"]    = (df[col] == 0)
        df[col] = df[col].fillna(-1)

    # ── 5. 로그 변환 & Winsor ---------------------------------
    for col in WINSOR_COLS:
        df[col] = np.log1p(df[col].fillna(0))

    if mode == "nn":
        # ↳ NN만 Winsor
        if is_train:
            clip_dict = {c: df[c].quantile(0.995) for c in WINSOR_COLS}
        for c in WINSOR_COLS:
            df[c] = df[c].clip(upper=clip_dict[c])

    # ── 6. 대칭성 비율 ────────────────────────────────────────
    df["pkt_ratio_fb"]  = np.log((df["pkt_count_fwd"]+EPS)/(df["pkt_count_bwd"]+EPS)).clip(-15, 15)
    df["byte_ratio_fb"] = np.log((df["rate_fwd_bytes"]+EPS)/(df["rate_bwd_bytes"]+EPS)).clip(-15, 15)

    # ── 7. 추가 파생 ──────────────────────────────────────────
    df["duration_log"] = df["duration"]               # (log값 자체를 보존)
    df["is_long_session"] = np.expm1(df["duration"]) > 30
    df["is_long_iat"]     = np.expm1(df["iat_avg_packets"]) > 360

    df["tcp_flag_total"] = df["tcp_syn_count"] + df["tcp_psh_count"] + df["tcp_rst_count"]
    df["is_syn_flood"]   = (df["tcp_syn_count"] > 3) & (df["pkt_count_bwd"] == 0)


    # 파생변수 추가
    # 파생 변수 1. 총 패킷 수 (송신 + 수신)
    df['pkt_count_total'] = df['pkt_count_fwd'] + df['pkt_count_bwd']

    # 파생 변수 2. 총 패킷 전송 속도 (송신 + 수신)
    df['rate_pkts_total'] = df['rate_fwd_pkts'] + df['rate_bwd_pkts']

    # 파생 변수 3. 패킷당 평균 페이로드 (송신)
    df['payload_fwd_per_pkt'] = df['payload_fwd_mean'] / (df['pkt_count_fwd'] + 1e-5)

    # 파생 변수 4. 패킷당 평균 페이로드 (수신)
    df['payload_bwd_per_pkt'] = df['payload_bwd_mean'] / (df['pkt_count_bwd'] + 1e-5)

    # 파생 변수 5. 전송 속도 × 시간 (송신 측 트래픽 총량 근사)
    df['rate_fwd_pkts_time_adj'] = df['rate_fwd_pkts'] * df['duration']

    # 파생 변수 6. 전송 속도 × 시간 (수신 바이트 기준 트래픽 총량 근사)
    df['rate_bwd_bytes_time_adj'] = df['rate_bwd_bytes'] * df['duration']


    # ── 8. 프로토콜 원-핫 ─────────────────────────────────────
    df = pd.get_dummies(df, columns=["protocol"], dummy_na=False)

    # ── 9. 불필요 원본 열 제거 ────────────────────────────────
    DROP = [
        "ID",               # ← 추가
        "attack_type",      # ← 추가 (train 데이터에만 존재)
        "ip_src", "ip_dst",
    ]
    # ── [NEW] 문자열 컬럼을 category로 캐스팅 ────────────────
    cat_cols = df.select_dtypes("object").columns.tolist()
    for c in cat_cols:
        df[c] = df[c].astype("category")
    df = df.drop(columns=[c for c in DROP if c in df.columns])

    # ── 10. Boolean → int 변환 ───────────────────────────────
    bool_cols = df.select_dtypes("bool").columns
    df[bool_cols] = df[bool_cols].astype("int8")

    # ── 11. NaN & Scaling 분기 ───────────────────────────────
    num_cols = (
        df.select_dtypes("number")
        .columns.difference(["target"])
    )

    if mode == "nn":
        #   a) 결측치 평균 대체
        if is_train:
            imputer = SimpleImputer(strategy="mean")
            df[num_cols] = imputer.fit_transform(df[num_cols])
            imp_stats = imputer.statistics_.tolist()
        else:
            imp_stats = scaler_params["imp_stats"]
            df[num_cols] = df[num_cols].fillna(
                pd.Series(imp_stats, index=num_cols)
            )

        #   b) StandardScaler
        if is_train:
            scaler = StandardScaler()
            df[num_cols] = scaler.fit_transform(df[num_cols])
            scaler_params = {
                "mean":  scaler.mean_.tolist(),
                "scale": scaler.scale_.tolist(),
                "cols":  num_cols.tolist(),
                "imp_stats": imp_stats,
            }
        else:
            mean  = np.array(scaler_params["mean"])
            scale = np.array(scaler_params["scale"])
            df[num_cols] = (df[num_cols] - mean) / scale

    else:  # mode == "tree"
        # 결측치 · 스케일 유지 (트리는 NaN 자체 활용)
        scaler_params = None
        if is_train and clip_dict is None:
            clip_dict = {c: None for c in WINSOR_COLS}

    # ── 12. 범주형(카테고리) 변수 원-핫 인코딩 (트리 모델용) ───────────────
        df = pd.get_dummies(df, columns=["port_src_bucket", "port_dst_bucket"], dummy_na=False)

    return df, clip_dict, scaler_params, lbl2id



In [139]:
# Train / Test ───────────────────────────────────────────────
train_tree, clip_tree, _, lbl2id = preprocess(train, mode="tree",  is_train=True)
train_nn,   clip_nn,  sp, _   = preprocess(train, mode="nn",    is_train=True)

test_tree, _, _, _  = preprocess(test,  mode="tree",  is_train=False,
                                        clip_dict=clip_tree)
test_nn, _,  _, _   = preprocess(test,  mode="nn",    is_train=False,
                                        clip_dict=clip_nn,
                                        scaler_params=sp)

# 저장 ───────────────────────────────────────────────────────
# train_tree.to_parquet(f"{OUT_DIR}/train_tree.parquet", index=False)
# test_tree.to_parquet (f"{OUT_DIR}/test_tree.parquet",  index=False)
# train_nn.to_parquet  (f"{OUT_DIR}/train_nn.parquet",   index=False)
# test_nn.to_parquet   (f"{OUT_DIR}/test_nn.parquet",    index=False)

print("✅ Parquet saved (tree & nn versions)")

✅ Parquet saved (tree & nn versions)


In [140]:
train_tree.head()

,port_src,port_dst,duration,pkt_count_fwd,pkt_count_bwd,rate_fwd_pkts,rate_bwd_pkts,rate_fwd_bytes,rate_bwd_bytes,payload_fwd_mean,...,protocol_TCP,protocol_UDP,port_src_bucket_missing,port_src_bucket_well,port_src_bucket_registered,port_src_bucket_dynamic,port_dst_bucket_missing,port_dst_bucket_well,port_dst_bucket_registered,port_dst_bucket_dynamic
0,3721.0,55.0,0.000231,2,2,9.066236,9.066236,12.467322,13.948923,81.000000,...,0,1,False,False,True,False,False,True,False,False
1,-1.0,91.0,0.000000,0,1,0.000000,0.000000,0.000000,0.000000,0.000000,...,1,0,True,False,False,False,False,True,False,False
2,-1.0,83.0,0.473748,11,5,2.952395,2.224710,7.455135,9.859254,790.125000,...,1,0,True,False,False,False,False,True,False,False
3,47668.0,-1.0,0.695060,6,6,1.942635,1.942635,5.786150,9.354594,993.416667,...,1,0,False,False,True,False,True,False,False,False
4,51753.0,451.0,5.205079,62,89,0.000000,0.399570,2.024592,6.537671,-1.000000,...,1,0,False,False,False,True,False,True,False,False


# 모델 생성

In [141]:
# 데이터 분리
from sklearn.model_selection import train_test_split
X = train_tree.drop(columns=["target"])
y = train_tree["target"]

#SET 분할
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# 증강
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

def augment_with_noise(X, y, noise_level=0.01, target_min_count=None):
    """
    Numeric-only noise injection augmenter.
    - X: pandas.DataFrame (feature set)
    - y: pandas.Series (labels)
    - noise_level: 전체 feature 스케일 대비 노이즈 비율
    - target_min_count: 각 클래스가 최소 이만큼 샘플을 갖도록 증강;
                        None이면 최빈 클래스 크기만큼 증강
    """
    # 1) 원본 보존
    X = X.copy()
    y = y.copy()

    # 2) 클래스별 카운트
    counts = Counter(y)
    max_count = max(counts.values()) if target_min_count is None else target_min_count

    # 3) 수치형 컬럼만 골라내기
    numeric_cols = X.select_dtypes(include=[np.number]).columns

    X_aug_list, y_aug_list = [], []

    for cls, cnt in counts.items():
        n_needed = max_count - cnt
        if n_needed <= 0:
            continue

        X_cls = X[y == cls]
        # feature별 범위 (max - min)
        feature_ranges = X_cls[numeric_cols].max() - X_cls[numeric_cols].min()

        for _ in range(n_needed):
            idx = np.random.randint(0, X_cls.shape[0])
            row = X_cls.iloc[idx].copy()

            base_vals = row[numeric_cols].values.astype(float)
            noise = np.random.normal(
                loc=0,
                scale=noise_level * feature_ranges.values,
                size=base_vals.shape
            )
            row[numeric_cols] = base_vals + noise

            X_aug_list.append(row)
            y_aug_list.append(cls)

    # 4) 원본 + 증강 결합
    if X_aug_list:
        X_aug = pd.DataFrame(X_aug_list, columns=X.columns)
        y_aug = pd.Series(y_aug_list, name=y.name)
        X_res = pd.concat([X, X_aug], ignore_index=True)
        y_res = pd.concat([y, y_aug], ignore_index=True)
    else:
        X_res, y_res = X, y

    print("Before augmentation:", dict(counts))
    print("After augmentation:", Counter(y_res))
    return X_res, y_res

# ────────────────────────────────────────────

# 2) 소수 클래스 증강
X_train_aug, y_train_aug = augment_with_noise(
    X_train, y_train,
    noise_level=0.01,       # 노이즈 세기 (0.01~0.05 사이 추천)
    target_min_count=200    # 원하는 최소 샘플 수
)

# # 3) RandomForest 학습
# rf = RandomForestClassifier(
#     n_estimators=300,
#     max_depth=15,
#     random_state=42,
#     class_weight='balanced_subsample',
#     n_jobs=-1
# )
# rf.fit(X_train_aug, y_train_aug)

# # 4) 검증
# y_pred = rf.predict(X_val)
# print(classification_report(y_val, y_pred))


Before augmentation: {4: 634, 2: 377, 0: 7033, 1: 1375, 9: 22, 7: 24, 10: 27, 3: 37, 5: 21, 11: 33, 8: 29, 6: 11}
After augmentation: Counter({0: 7033, 1: 1375, 4: 634, 2: 377, 9: 200, 7: 200, 10: 200, 3: 200, 5: 200, 11: 200, 8: 200, 6: 200})


In [142]:
# y_train.value_counts()

In [143]:
# !pip install catboost

In [144]:
from sklearn.ensemble import ExtraTreesClassifier

# 최적 하이퍼 파라미터:  {'max_depth': 12, 'min_samples_leaf': 8, 'min_samples_split': 8, 'n_estimators': 700}

model_et = ExtraTreesClassifier(
    class_weight={6:10, 3:2},       # 클래스 6에 10배, 클래스 3에 2배 가중치
    n_estimators=700,
    max_depth=15,
    random_state=42,
    min_samples_leaf= 8,
    min_samples_split= 8,
    n_jobs=-1
)

"""
# source code
n_estimators=100,
criterion='gini',
max_depth=None,
min_samples_split=2,
min_samples_leaf=1,
min_weight_fraction_leaf=0.0,
max_features='sqrt',
max_leaf_nodes=None,
min_impurity_decrease=0.0,
bootstrap=False,
oob_score=False,
n_jobs=None,
random_state=None,
verbose=0,
warm_start=False,
class_weight=None,
ccp_alpha=0.0,
max_samples=None

"""
model_et.fit(X_train_aug, y_train_aug)
y_pred_et = model_et.predict(X_val)

In [145]:
# 평가 리포트 출력
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
print("ExtraTree Report:")
print(classification_report(y_val, y_pred_et))

ExtraTree Report:
              precision    recall  f1-score   support

           0       0.99      1.00      0.99      1758
           1       0.99      0.97      0.98       344
           2       0.99      0.98      0.98        94
           3       1.00      0.80      0.89        10
           4       1.00      0.99      0.99       159
           5       1.00      1.00      1.00         5
           6       0.33      0.67      0.44         3
           7       1.00      0.83      0.91         6
           8       0.80      0.57      0.67         7
           9       1.00      0.80      0.89         5
          10       1.00      0.57      0.73         7
          11       0.86      0.75      0.80         8

    accuracy                           0.99      2406
   macro avg       0.91      0.83      0.86      2406
weighted avg       0.99      0.99      0.99      2406



In [146]:
# from sklearn.model_selection import GridSearchCV

# params = {
#     'n_estimators':[200, 500, 700],
#     'max_depth' : [6, 8, 10, 12, 15, 18],
#     'min_samples_leaf' : [8, 12, 18],
#     'min_samples_split' : [8, 16, 20]
# }

# rf_clf = ExtraTreesClassifier(random_state=0, n_jobs=-1)
# grid_cv = GridSearchCV(rf_clf , param_grid=params , cv=2, n_jobs=2)
# grid_cv.fit(X_train_aug , y_train_aug) # grid.cv.fit(train_x, train_y)

# estimator =grid_cv.best_estimator_
# pred = estimator.predict(X_val) # estimator.predict(test)

# print('최적 하이퍼 파라미터:\n', grid_cv.best_params_)
# print('최고 예측 정확도: {0:.4f}'.format(grid_cv.best_score_))

In [147]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    random_state=42,
    n_jobs=-1,
    min_samples_leaf = 8,
    class_weight='balanced_subsample'  # 불균형 대응
)

"""
class sklearn.ensemble.RandomForestClassifier(
    n_estimators=100,
    max_features='sqrt',
    criterion='gini',
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    min_weight_fraction_leaf=0.0,
    max_leaf_nodes=None,
    min_impurity_decrease=0.0,
    bootstrap=True,
    n_jobs=None,
    random_state=None,
    verbose=0,
    class_weight=None
)
"""

model_rf.fit(X_train_aug, y_train_aug)
y_pred_rf = model_rf.predict(X_val)

In [148]:

# 평가 리포트 출력
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
print("RandomForest Report:")
print(classification_report(y_val, y_pred_rf))

RandomForest Report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00      1758
           1       0.98      0.99      0.99       344
           2       1.00      1.00      1.00        94
           3       0.77      1.00      0.87        10
           4       1.00      0.99      0.99       159
           5       0.83      1.00      0.91         5
           6       0.00      0.00      0.00         3
           7       1.00      1.00      1.00         6
           8       0.70      1.00      0.82         7
           9       1.00      1.00      1.00         5
          10       1.00      0.43      0.60         7
          11       0.70      0.88      0.78         8

    accuracy                           0.99      2406
   macro avg       0.83      0.86      0.83      2406
weighted avg       0.99      0.99      0.99      2406



In [150]:
!pip install lightgbm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 37.9 MB/s eta 0:00:00


In [155]:
import lightgbm as lgb

# 1) 모델 정의
model_lgb = lgb.LGBMClassifier(
    objective='multiclass',
    num_class=len(y_train_aug.unique()),
    learning_rate=0.05,
    n_estimators=500,
    max_depth=10,
    num_leaves=64,
    class_weight={6:10},        # 클래스 6에 10배 가중치
    random_state=42,
    n_jobs=-1
)

model_lgb.fit(
    X_train_aug,
    y_train_aug,
)

y_pred_lgb = model_lgb.predict(X_val)


# 평가 리포트 출력
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
print("lgbm Report:")
print(classification_report(y_val, y_pred_lgb))

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003585 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11775
[LightGBM] [Info] Number of data points in the train set: 11019, number of used features: 62
[LightGBM] [Info] Start training from score -0.600315
[LightGBM] [Info] Start training from score -2.232475
[LightGBM] [Info] Start training from score -3.526439
[LightGBM] [Info] Start training from score -4.160366
[LightGBM] [Info] Start training from score -3.006635
[LightGBM] [Info] Start training from score -4.160366
[LightGBM] [Info] Start training from score -1.857781
[LightGBM] [Info] Start training from score -4.160366
[LightGBM] [Info] Start training from score -4.160366
[LightGBM] [Info] Start training from score -4.160366
[LightGBM] [Info] Start training from score -4.160366
[LightGBM] [Info] Start training from score -4.160366
[LightGBM] [Warning] No further splits with positive gain, bes

In [156]:
# 소프트 보팅

from sklearn.ensemble import VotingClassifier

voting_clf = VotingClassifier(
    estimators=[
        ('ExtraTrees', model_et),
        ('RandomForest', model_rf),
        ('lgbm', model_lgb)
    ],
    voting='soft'
)

voting_clf.fit(X_train_aug, y_train_aug)
y_pred_vote = voting_clf.predict(X_val)

print("VotingClassifier Report:")
print(classification_report(y_val, y_pred_vote))


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003450 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 11775
[LightGBM] [Info] Number of data points in the train set: 11019, number of used features: 62
[LightGBM] [Info] Start training from score -0.600315
[LightGBM] [Info] Start training from score -2.232475
[LightGBM] [Info] Start training from score -3.526439
[LightGBM] [Info] Start training from score -4.160366
[LightGBM] [Info] Start training from score -3.006635
[LightGBM] [Info] Start training from score -4.160366
[LightGBM] [Info] Start training from score -1.857781
[LightGBM] [Info] Start training from score -4.160366
[LightGBM] [Info] Start training from score -4.160366
[LightGBM] [Info] Start training from score -4.160366
[LightGBM] [Info] Start training from score -4.160366
[LightGBM] [Info] Start training from score -4.160366
[LightGBM] [Warning] No further splits with positive gain, bes

In [157]:
# from sklearn.model_selection import GridSearchCV

# params = {
#     'n_estimators':[200, 500, 700],
#     'max_depth' : [6, 8, 10, 12],
#     'min_samples_leaf' : [8, 12, 18],
#     'min_samples_split' : [8, 16, 20]
# }

# rf_clf = RandomForestClassifier(random_state=0, n_jobs=-1)
# grid_cv = GridSearchCV(rf_clf , param_grid=params , cv=2, n_jobs=2)
# grid_cv.fit(X_train , y_train) # grid.cv.fit(train_x, train_y)

# estimator =grid_cv.best_estimator_
# pred = estimator.predict(X_val) # estimator.predict(test)

# print('최적 하이퍼 파라미터:\n', grid_cv.best_params_)
# print('최고 예측 정확도: {0:.4f}'.format(grid_cv.best_score_))

# 결과 출력

In [158]:
# test 전처리 데이터 이름 및 ID 인덱스
"""
test_ids : ID 컬럼
attack_type : 라벨

"""

# 앞서 준비한 "10. 앙상블 준비"를 통해 예측
y_final = voting_clf.predict(test_tree)

# 디코딩
id2lbl = {v:k for k,v in lbl2id.items()}
y_final_label = [id2lbl[i] for i in y_final]

# submission
submission = pd.DataFrame({
    'ID': test_ids,
    'attack_type': y_final_label
})
# 생성
submission.to_csv('submission_5.csv', index=False)

#체크
print(submission.head())


          ID    attack_type
0  TEST_0000         Benign
1  TEST_0001  Port_Scanning
2  TEST_0002         Benign
3  TEST_0003         Benign
4  TEST_0004         Benign
